# safety-sae-feature-forecasting: Step 1 - smoke test

Verifies the full activation-extraction pipeline:

1. Load Gemma-2-2B (base) via `transformer_lens`
2. Load a GemmaScope residual-stream SAE
3. Forward pass with cached residual streams at chosen layers
4. Encode the late-layer residual through the SAE and inspect top-activating features per token

**Runtime:** Runtime → Change runtime type → **T4 GPU**.

**Auth:** Gemma-2-2B is gated. Before running:
1. Accept the license at https://huggingface.co/google/gemma-2-2b
2. Generate a read token at https://huggingface.co/settings/tokens
3. Paste the token in the login cell below

Step 1 is complete when the final cell prints non-zero SAE feature activations per token.

## 1. Install dependencies

Takes ~1-2 min on a fresh Colab runtime.

In [ ]:
!pip install -q transformer_lens sae_lens

## 2. Authenticate with Hugging Face

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Load Gemma-2-2B

Loads in bf16 on the T4 (~5 GB VRAM). Takes ~1-2 min.

In [ ]:
import torch
from transformer_lens import HookedTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cpu':
    print('WARNING: no GPU detected. Switch runtime to T4 GPU before continuing.')

model = HookedTransformer.from_pretrained(
    'gemma-2-2b',
    device=device,
    dtype=torch.bfloat16,
)
model.eval()
print(f'Loaded. n_layers={model.cfg.n_layers}, d_model={model.cfg.d_model}')

## 4. Load a GemmaScope SAE (layer 20, residual stream, width 16k)

If the `release` / `sae_id` strings below don't match the registry, the next cell prints what's available so you can adjust.

In [ ]:
from sae_lens import SAE

RELEASE = 'gemma-scope-2b-pt-res-canonical'
SAE_ID = 'layer_20/width_16k/canonical'

try:
    sae, cfg_dict, sparsity = SAE.from_pretrained(
        release=RELEASE,
        sae_id=SAE_ID,
        device=device,
    )
    print(f'SAE loaded. d_in={sae.cfg.d_in}, d_sae={sae.cfg.d_sae}, dtype={sae.W_enc.dtype}')
    print(f'hook_name={sae.cfg.hook_name}')
except Exception as e:
    print(f'Load failed: {e}')
    print()
    print('Available Gemma Scope releases:')
    from sae_lens.toolkit.pretrained_saes_directory import get_pretrained_saes_directory
    directory = get_pretrained_saes_directory()
    for name in sorted(directory.keys()):
        if 'gemma-scope' in name.lower() and '2b' in name.lower():
            saes = list(directory[name].saes_map.keys())
            print(f'  {name}: {len(saes)} SAEs (first few: {saes[:3]})')
    raise

## 5. Forward pass with cached residual streams

We cache only the two hooks we need (layer 5 + layer 20 post-residual) to keep memory small.

In [ ]:
prompt = "The customer's complaint, while rude, raises a valid point."
tokens = model.to_tokens(prompt)
print(f'Token count: {tokens.shape[1]}')
print(f'Tokens: {model.to_str_tokens(prompt)}')

wanted = {'blocks.5.hook_resid_post', 'blocks.20.hook_resid_post'}

with torch.no_grad():
    _, cache = model.run_with_cache(tokens, names_filter=lambda n: n in wanted)

print(f'cache keys: {list(cache.keys())}')
print(f'layer 5 resid shape:  {cache["blocks.5.hook_resid_post"].shape}')
print(f'layer 20 resid shape: {cache["blocks.20.hook_resid_post"].shape}')

## 6. Encode the layer-20 residual through the SAE

Smoke-test success criterion: for each token, a small fraction of the ~16k features are active (typical L0 for GemmaScope canonical SAEs is ~50-200 per token), and the top features have sane-looking magnitudes.

In [ ]:
resid_20 = cache['blocks.20.hook_resid_post'].to(sae.W_enc.dtype)

with torch.no_grad():
    feature_acts = sae.encode(resid_20)

print(f'feature_acts shape: {tuple(feature_acts.shape)}  # (batch, seq, d_sae)')
print()

str_tokens = model.to_str_tokens(prompt)
for tok_idx in range(feature_acts.shape[1]):
    acts = feature_acts[0, tok_idx].float()
    nonzero = int((acts > 0).sum().item())
    top = torch.topk(acts, k=5)
    top_pairs = [(int(i), round(float(v), 3)) for i, v in zip(top.indices, top.values)]
    print(f'{tok_idx:>3}  {repr(str_tokens[tok_idx]):<14}  active={nonzero:>4}  top-5: {top_pairs}')

## Done

If the cell above printed feature activations for every token (with ~tens to hundreds of non-zero features each), the pipeline works end-to-end.

**Next step (Step 2):** browse Neuronpedia at https://www.neuronpedia.org/gemma-2-2b/20-gemmascope-res-16k to identify safety-flavored target features. You'll bring back 3-5 feature indices for use in Step 3 (full activation cache extraction).